# 💊 알약 감지 베이스라인 (Healthcare Object Detection)

**목표**: 이미지 속 최대 4개의 알약의 클래스와 바운딩 박스를 검출  
**모델**: Faster R-CNN (ResNet-50-FPN v2)  
**평가**: mAP@0.5

```
pill_detection/
├── configs/config.yaml
├── data/raw/images/ & annotations/
├── src/ (dataset, model, train, evaluate, predict, utils)
└── outputs/ (checkpoints, logs, predictions)
```

## 0. 환경 설정

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pandas as pd
from src.utils import load_config, set_seed, get_device

config = load_config('../configs/config.yaml')
set_seed(config['project']['seed'])
device = get_device()
CLASSES = config['data']['classes']
print('클래스:', CLASSES)

## 1. 데이터 탐색 (EDA)

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

IMAGE_DIR = config['data']['image_dir']
ANNO_DIR  = config['data']['annotation_dir']

records = []
for xml_file in Path(ANNO_DIR).glob('*.xml'):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    for obj in root.findall('object'):
        bnd = obj.find('bndbox')
        records.append({
            'image': xml_file.stem,
            'class': obj.find('name').text,
            'xmin': int(float(bnd.find('xmin').text)),
            'ymin': int(float(bnd.find('ymin').text)),
            'xmax': int(float(bnd.find('xmax').text)),
            'ymax': int(float(bnd.find('ymax').text)),
        })

df = pd.DataFrame(records)
print(f'어노테이션 수: {len(df)}, 이미지 수: {df["image"].nunique()}')
df.head()

In [ ]:
# 클래스 분포 & 이미지당 알약 수
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['class'].value_counts().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('클래스별 어노테이션 수')
axes[0].tick_params(axis='x', rotation=30)

df.groupby('image').size().hist(bins=range(1, 6), rwidth=0.8, ax=axes[1], color='coral')
axes[1].set_title('이미지당 알약 수 분포')
axes[1].set_xlabel('알약 수')

plt.tight_layout()
plt.show()

In [ ]:
# Ground Truth 시각화 (샘플 4장)
import matplotlib.patches as patches

COLORS = ['#FF4444', '#4CAF50', '#2196F3', '#FF9800']
sample_names = df['image'].unique()[:4]
fig, axes = plt.subplots(1, len(sample_names), figsize=(5*len(sample_names), 5))

for ax, name in zip(axes, sample_names):
    img = cv2.cvtColor(cv2.imread(os.path.join(IMAGE_DIR, f'{name}.jpg')), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    rows = df[df['image'] == name]
    for i, (_, row) in enumerate(rows.iterrows()):
        color = COLORS[i % 4]
        ax.add_patch(patches.Rectangle(
            (row.xmin, row.ymin), row.xmax-row.xmin, row.ymax-row.ymin,
            linewidth=2.5, edgecolor=color, facecolor='none'
        ))
        ax.text(row.xmin, row.ymin-8, row['class'], color='white', fontsize=9,
                bbox=dict(facecolor=color, alpha=0.8, pad=2, edgecolor='none'))
    ax.set_title(f'{name} ({len(rows)}개)')
    ax.axis('off')

plt.suptitle('Ground Truth 시각화', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. 데이터셋 준비

In [ ]:
from src.dataset import build_dataloaders

train_loader, val_loader, test_loader, classes = build_dataloaders(config)
print(f'Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

# 배치 확인
images, targets = next(iter(train_loader))
print(f'이미지 Shape: {images[0].shape}')
print(f'타겟 예시 - boxes: {targets[0]["boxes"].shape}, labels: {targets[0]["labels"]}')

## 3. 모델 준비

In [ ]:
from src.model import build_model, count_parameters

model = build_model(config).to(device)
count_parameters(model)

## 4. 학습

In [ ]:
from src.train import train
train(config)

# 터미널에서 실행하려면:
# !python src/train.py --config configs/config.yaml

## 5. 학습 로그 시각화

In [ ]:
log_df = pd.read_csv(os.path.join(config['output']['log_dir'], 'train_log.csv'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(log_df['epoch'], log_df['train_loss'], marker='o', color='coral')
axes[0].set(title='Train Loss', xlabel='Epoch', ylabel='Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(log_df['epoch'], log_df['mAP'], marker='o', color='steelblue')
axes[1].set(title='Validation mAP@0.5', xlabel='Epoch', ylabel='mAP')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best = log_df.loc[log_df['mAP'].idxmax()]
print(f'최고 성능 → Epoch {int(best["epoch"])}: mAP@0.5 = {best["mAP"]:.4f}')

## 6. 추론 및 시각화

In [ ]:
from src.utils import load_checkpoint, visualize_prediction

ckpt_path = os.path.join(config['output']['checkpoint_dir'], 'best_model.pth')
load_checkpoint(model, None, ckpt_path, device)

model.eval()
MAX_VIS = 6
count = 0

with torch.no_grad():
    for images, image_names in test_loader:
        preds = model([img.to(device) for img in images])
        for img, pred, name in zip(images, preds, image_names):
            if count >= MAX_VIS: break
            visualize_prediction(
                img, pred, classes,
                score_threshold=config['model']['score_threshold'],
                max_detections=config['model']['max_detections'],
                title=name
            )
            count += 1
        if count >= MAX_VIS: break

## 7. 성능 개선 방향

| 방향 | 방법 |
|---|---|
| 데이터 증강 | Mosaic, MixUp, CutMix, 색상 지터링 |
| 모델 교체 | YOLOv8, DINO, Co-DETR |
| 하이퍼파라미터 | LR Warmup, Cosine Annealing |
| 앙상블 | WBF (Weighted Box Fusion) |
| 클래스 불균형 | Focal Loss, 오버샘플링 |
| 해상도 향상 | 800 → 1024 이상 |